|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Guided decoding<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: make invalid output unreachable<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np

from tests.helpers import json_prefix_state

rng = np.random.default_rng(0)
# a vocabulary of multi-character pieces, like a real tokenizer
VOCAB = ['{', '}', '[', ']', '"', ':', ',', ' ', '1', '2', '42',
         'true', 'false', 'null', 'name', 'age', '": ', '", "', ': {', '}, ']
print(f'{len(VOCAB)} tokens, and most of them are more than one character')

Constrain the output so that invalid JSON is not merely unlikely but
unreachable.

The grammar is given to you: `json_prefix_state` classifies a string as
`valid`, `prefix` or `invalid`. Your job is the part that touches the
sampler, and the part that makes it affordable.

# Exercise 1: which tokens are legal here?

Note that the vocabulary contains multi-character pieces like `'": '`. That
is what makes real guided decoding fiddly: the FSM advances by a whole token,
not a character.

In [ ]:
def legal(prefix):
  """Which tokens can follow `prefix` without making it unrecoverable?

  A token is legal when appending its WHOLE string leaves the prefix in
  'prefix' or 'valid'. Not one character: the whole token."""
  return 

for p in ['', '{', '{"', '{"name', '{"name"', '{"name": ']:
  m = legal(p)
  print(f'{p!r:<12} {m.sum():>2}/{len(VOCAB)}: {[t for t,k in zip(VOCAB,m) if k][:8]}')

# Exercise 2: sample with the mask on

Two hundred runs with random logits. Not one of them should produce invalid
JSON.

In [ ]:
def masked_sample(prefix, logits):
  """Sample a token, but only from the legal ones."""
  m = legal(prefix)
  if not m.any(): return None          # dead end, see Exercise 3
  # set the illegal logits to -inf, softmax, draw
  masked = 
  p = 
  return VOCAB[int(rng.choice(len(VOCAB), p=p))]

bad = 0
for trial in range(200):
  prefix = ''
  for _ in range(14):
    t = masked_sample(prefix, rng.normal(size=len(VOCAB)))
    if t is None: break
    prefix += t
    if json_prefix_state(prefix) == 'valid' and len(prefix) > 6: break
  if json_prefix_state(prefix) == 'invalid': bad += 1
print(f'{bad}/200 samples produced invalid JSON')
print(f'example: {prefix!r}  ({json_prefix_state(prefix)})')

# Exercise 3: where the one-step mask is not enough

Count the runs that reach a prefix with no legal continuation at all.

In [ ]:
# The mask guarantees you never become INVALID. Does it guarantee you
# finish? Run to a length limit, as a server does, and classify the
# final state of each run.
states = {'valid':0, 'prefix':0, 'invalid':0}
examples = []
for trial in range(500):
  prefix = ''
  for step in range(14):
    
  s = json_prefix_state(prefix)
  

for k, v in states.items():
  print(f'{k:>8}: {v:>4}/500')
print('\nincomplete examples, stopped by the length limit:')
for e in examples: print(f'  {e!r}')

### Before you open the solution

1. Exercise 2 gives zero invalid outputs. Exercise 3 should show that
   most runs end in `prefix` rather than `valid`. State precisely what
   the mask guarantees and what it does not.
2. What would the masker need to know in order to guarantee completion
   as well? Why can a string-prefix validator not provide it cheaply?
3. Your `legal()` runs the validator once per vocabulary entry. At
   151,936 tokens and 2 microseconds each, what does one masked step
   cost against a 10 ms decode step? What has to be cached, and what is
   the cache keyed on?